# Licence Plate Detector: YOLO26m + PaddleOCR (CCPD2019)

Notebook-first refactor for plate detection plus OCR using a real Kaggle dataset download and verifiable outputs.

## 1. Problem Type and Why This Method

Task family: plate detection + OCR.

- YOLO26m is used for plate localization.
- PaddleOCR is used for text reading from detected plate crops.
- This separation keeps detection and OCR evaluation honest and interpretable.

## 2. Dataset Source and License Note

Dataset: https://www.kaggle.com/datasets/hrishirajdevsarmah/ccpd2019

Use this dataset according to Kaggle dataset terms and the dataset author's license notes.

## 3. Environment Setup

In [1]:
import os
import platform
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Python: {platform.python_version()}')
print(f'PyTorch: {torch.__version__}')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Python: 3.13.12
PyTorch: 2.6.0+cu124
Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 4. Install Dependencies

In [2]:
import importlib
import subprocess
import sys

def ensure_package(import_name, pip_name=None):
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

ensure_package('ultralytics')
ensure_package('kagglehub')
ensure_package('cv2', 'opencv-python')
ensure_package('matplotlib')
ensure_package('pandas')
ensure_package('paddleocr')
print('Dependencies available.')

e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(
e:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


Dependencies available.


## 5. Imports and Paths

In [3]:
import json
import shutil
from pathlib import Path

import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO
from paddleocr import PaddleOCR

PROJECT_DIR = Path.cwd()
WORK_DIR = PROJECT_DIR / 'working'
DATA_DIR = WORK_DIR / 'data'
RUNS_DIR = PROJECT_DIR / 'runs'
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'

for d in [WORK_DIR, DATA_DIR, RUNS_DIR, ARTIFACTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

KAGGLE_DATASET = 'hrishirajdevsarmah/ccpd2019'
print(f'Project dir: {PROJECT_DIR}')

Project dir: e:\Github\Machine-Learning-Projects\Computer Vision\Licence Plate Detector\Source Code


## 6. Real Dataset Download (Kaggle CCPD2019)

In [ ]:
def check_kaggle_credentials():
    has_env = bool(os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'))
    has_file = (Path.home() / '.kaggle' / 'kaggle.json').exists()
    if not has_env and not has_file:
        raise RuntimeError('Kaggle credentials missing. Set KAGGLE_USERNAME/KAGGLE_KEY or create ~/.kaggle/kaggle.json')

def download_ccpd():
    check_kaggle_credentials()
    try:
        import kagglehub
        return Path(kagglehub.dataset_download(KAGGLE_DATASET))
    except Exception:
        target = DATA_DIR / 'ccpd2019_raw'
        target.mkdir(parents=True, exist_ok=True)
        subprocess.check_call(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', str(target), '--unzip'])
        return target

raw_root = Path(download_ccpd())
print(f'Raw dataset path: {raw_root}')

## 7. Verify Files, Labels, and Splits

In [ ]:
def find_yolo_dataset_root(root):
    candidates = [root]
    if root.exists():
        for child in root.iterdir():
            if child.is_dir():
                candidates.append(child)
    for candidate in candidates:
        data_yaml = candidate / 'data.yaml'
        train_images = candidate / 'train' / 'images'
        val_images = candidate / 'valid' / 'images'
        alt_val_images = candidate / 'val' / 'images'
        train_labels = candidate / 'train' / 'labels'
        val_labels = candidate / 'valid' / 'labels'
        alt_val_labels = candidate / 'val' / 'labels'
        has_train = train_images.exists() and train_labels.exists()
        has_valid = (val_images.exists() and val_labels.exists()) or (alt_val_images.exists() and alt_val_labels.exists())
        if data_yaml.exists() and has_train and has_valid:
            return candidate
    raise RuntimeError('Could not find YOLO-formatted train/valid dataset structure in CCPD2019 download.')

YOLO_ROOT = find_yolo_dataset_root(raw_root)
DATA_YAML = YOLO_ROOT / 'data.yaml'
TRAIN_IMAGES = YOLO_ROOT / 'train' / 'images'
TRAIN_LABELS = YOLO_ROOT / 'train' / 'labels'
VAL_IMAGES = YOLO_ROOT / 'valid' / 'images' if (YOLO_ROOT / 'valid' / 'images').exists() else YOLO_ROOT / 'val' / 'images'
VAL_LABELS = YOLO_ROOT / 'valid' / 'labels' if (YOLO_ROOT / 'valid' / 'labels').exists() else YOLO_ROOT / 'val' / 'labels'
TEST_IMAGES = YOLO_ROOT / 'test' / 'images'
TEST_LABELS = YOLO_ROOT / 'test' / 'labels'

def count_files(path):
    return len([p for p in path.iterdir() if p.is_file()])

n_train_img = count_files(TRAIN_IMAGES)
n_train_lbl = count_files(TRAIN_LABELS)
n_val_img = count_files(VAL_IMAGES)
n_val_lbl = count_files(VAL_LABELS)

assert n_train_img > 0, 'No train images found'
assert n_train_img == n_train_lbl, f'Train mismatch: images={n_train_img}, labels={n_train_lbl}'
assert n_val_img > 0, 'No validation images found'
assert n_val_img == n_val_lbl, f'Validation mismatch: images={n_val_img}, labels={n_val_lbl}'

if TEST_IMAGES.exists() and TEST_LABELS.exists():
    n_test_img = count_files(TEST_IMAGES)
    n_test_lbl = count_files(TEST_LABELS)
    print(f'Test split: images={n_test_img}, labels={n_test_lbl}')

print(f'YOLO dataset root: {YOLO_ROOT}')
print(f'Train split: images={n_train_img}, labels={n_train_lbl}')
print(f'Val split: images={n_val_img}, labels={n_val_lbl}')

## 8. Sample Data Visualization

In [ ]:
sample_paths = sorted([p for p in TRAIN_IMAGES.iterdir() if p.is_file()])[:9]
fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, image_path in zip(axes.flatten(), sample_paths):
    label_path = TRAIN_LABELS / f'{image_path.stem}.txt'
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    if label_path.exists():
        rows = [line for line in label_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        for row in rows:
            cls, cx, cy, bw, bh = row.split()
            cx = float(cx) * w
            cy = float(cy) * h
            bw = float(bw) * w
            bh = float(bh) * h
            x1 = int(cx - bw / 2)
            y1 = int(cy - bh / 2)
            x2 = int(cx + bw / 2)
            y2 = int(cy + bh / 2)
            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 80, 80), 2)
    ax.imshow(image)
    ax.axis('off')
for ax in axes.flatten()[len(sample_paths):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'sample_training_labels.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Train YOLO26m Plate Detector

In [ ]:
EPOCHS = 20
IMGSZ = 960
BATCH = 8 if DEVICE == 'cuda' else 4
PRIMARY_MODEL = 'yolo26m.pt'
FALLBACK_MODEL = 'yolo26s.pt'

selected_model = PRIMARY_MODEL
detector = YOLO(selected_model)

try:
    detector.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, project=str(RUNS_DIR), name='licence_plate_detector', exist_ok=True, device=0 if DEVICE == 'cuda' else 'cpu', seed=SEED, workers=2, pretrained=True, verbose=True)
except RuntimeError as exc:
    if 'out of memory' not in str(exc).lower():
        raise
    selected_model = FALLBACK_MODEL
    detector = YOLO(selected_model)
    detector.train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=max(2, BATCH // 2), project=str(RUNS_DIR), name='licence_plate_detector', exist_ok=True, device=0 if DEVICE == 'cuda' else 'cpu', seed=SEED, workers=2, pretrained=True, verbose=True)

run_dir = Path(detector.trainer.save_dir)
best_pt = run_dir / 'weights' / 'best.pt'
assert best_pt.exists(), f'Missing trained weights: {best_pt}'
print(f'Selected model: {selected_model}')
print(f'Run directory: {run_dir}')

## 10. Detection Evaluation

In [ ]:
best_detector = YOLO(str(best_pt))
split_for_eval = 'test' if TEST_IMAGES.exists() else 'val'
val_result = best_detector.val(data=str(DATA_YAML), split=split_for_eval, imgsz=IMGSZ, batch=max(1, BATCH // 2), device=0 if DEVICE == 'cuda' else 'cpu', verbose=False)

metrics = {
    'dataset': KAGGLE_DATASET,
    'selected_model': selected_model,
    'epochs': EPOCHS,
    'imgsz': IMGSZ,
    'eval_split': split_for_eval,
    'precision': float(val_result.box.mp),
    'recall': float(val_result.box.mr),
    'map50': float(val_result.box.map50),
    'map50_95': float(val_result.box.map)
}
metrics_path = ARTIFACTS_DIR / 'metrics.json'
metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print(json.dumps(metrics, indent=2))

## 11. OCR on Detected Plates (PaddleOCR)

In [ ]:
ocr = PaddleOCR(use_angle_cls=True, lang='en', use_gpu=(DEVICE == 'cuda'))
infer_images = sorted([p for p in VAL_IMAGES.iterdir() if p.is_file()])[:10]
ocr_records = []
preview_rows = []

for img_path in infer_images:
    pred = best_detector.predict(source=str(img_path), conf=0.25, verbose=False)[0]
    rendered = pred.plot()
    rendered_rgb = cv2.cvtColor(rendered, cv2.COLOR_BGR2RGB)
    boxes = pred.boxes.xyxy.cpu().numpy() if pred.boxes is not None else np.empty((0, 4))
    texts = []
    src = cv2.imread(str(img_path))
    for box in boxes:
        x1, y1, x2, y2 = [int(v) for v in box]
        crop = src[max(0, y1):max(1, y2), max(0, x1):max(1, x2)]
        if crop.size == 0:
            continue
        ocr_out = ocr.ocr(crop, cls=True)
        if ocr_out and ocr_out[0]:
            parts = [item[1][0] for item in ocr_out[0] if item and len(item) > 1]
            if parts:
                texts.append(' '.join(parts))
    ocr_records.append({'image': img_path.name, 'num_detections': int(len(boxes)), 'ocr_texts': texts})
    preview_rows.append(rendered_rgb)

ocr_json = ARTIFACTS_DIR / 'ocr_predictions.json'
ocr_json.write_text(json.dumps(ocr_records, indent=2), encoding='utf-8')

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, frame in zip(axes.flatten(), preview_rows[:6]):
    ax.imshow(frame)
    ax.axis('off')
for ax in axes.flatten()[len(preview_rows[:6]):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(ARTIFACTS_DIR / 'detection_ocr_preview.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'OCR prediction file saved: {ocr_json}')

## 12. Error Analysis and Honest Limitations

Common failure modes:
- Motion blur and angled plates reduce OCR quality.
- Low-light images can reduce both detection and recognition.
- OCR language model can misread region-specific plate formats.

This notebook reports detector metrics quantitatively and OCR results qualitatively unless ground-truth OCR strings are available in parsed labels.

## 13. Save Artifacts

In [ ]:
onnx_out = Path(best_detector.export(format='onnx'))
final_best_pt = ARTIFACTS_DIR / 'best.pt'
final_best_onnx = ARTIFACTS_DIR / 'best.onnx'
shutil.copy2(best_pt, final_best_pt)
shutil.copy2(onnx_out, final_best_onnx)

manifest = {
    'project': 'Licence Plate Detector',
    'dataset': KAGGLE_DATASET,
    'stack': 'YOLO26m + PaddleOCR',
    'selected_model': selected_model,
    'artifacts': {
        'best_pt': str(final_best_pt),
        'best_onnx': str(final_best_onnx),
        'metrics_json': str(metrics_path),
        'ocr_predictions_json': str(ARTIFACTS_DIR / 'ocr_predictions.json'),
        'preview_png': str(ARTIFACTS_DIR / 'detection_ocr_preview.png'),
        'label_preview_png': str(ARTIFACTS_DIR / 'sample_training_labels.png')
    }
}
manifest_path = ARTIFACTS_DIR / 'artifact_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(json.dumps(manifest, indent=2))

## 14. Final Summary

This notebook performs real dataset download, split verification, YOLO26m plate detection training/evaluation, PaddleOCR plate reading on detections, and exports only real generated artifacts.